# Experimento 2 - Variacao de carga

Objetivo: analisar como cada politica (`threshold`, `on_demand`, `hybrid`) se comporta quando a demanda cresce.

Este experimento varia intensidade de requisicoes por tres perfis:
- carga baixa
- carga media
- carga alta

E observa:
- taxa de atendimento = `served_requests / (served_requests + denied_requests)`
- eficiencia de uso = `total_consumed_bits / total_generated_bits`
- pressao sobre o sistema = `replenishment_events`
- escassez residual = `bits_available` final

In [7]:
import sys
import os
import random
import importlib
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '/home/esdras/Wqnets/QuantumNet')

# Força reload do módulo (útil quando você editou o código e o kernel já estava rodando)
import quantumnet.experiments.qkd_policy_experiments as _qkdexp
import quantumnet.experiments as _qexp
importlib.reload(_qkdexp)
importlib.reload(_qexp)

from quantumnet.experiments import (
    DEFAULT_POLICIES,
    stable_seed,
    run_trials,
    aggregate_trials,
    write_csv,
 )

print('Imports carregados com sucesso')

Imports carregados com sucesso


## Configuracao do experimento

Topologia fixa: `Linha` com 4 nos e 3 enlaces diretos (`(0,1)`, `(1,2)`, `(2,3)`).

Para cada combinacao de politica e nivel de carga, executamos varias repeticoes para suavizar variacao estocastica do BB84.

In [ ]:
policies = list(DEFAULT_POLICIES)
link_pairs = [(0, 1), (1, 2), (2, 3)]

# Parâmetros estruturais (fixos em todos os perfis)
minimum_stock_bits = 96
buffer_capacity_bits = 512

# Modo validação (rápido)
trials_per_setting = 5
base_seed = 20260326

# Perfis de carga dentro do mesmo bloco experimental:
# - moderada: demanda equilibrada
# - sobrecarga: demanda persistentemente maior
# - burst: mesma ordem de grandeza de demanda média, mas concentrada em janelas curtas
load_profiles = {
    'moderada': {
        'mode': 'uniform',
        'requests_per_link': 30,
        'bit_options': [24, 32, 48, 64],
    },
    'sobrecarga': {
        'mode': 'uniform',
        'requests_per_link': 90,
        'bit_options': [48, 64, 96, 128],
    },
    'burst': {
        'mode': 'burst_windows',
        'duration_slots': 30,
        'burst_windows': [(5, 10), (15, 18)],  # (inicio, fim) por timeslot
        'burst_rate': 6,  # requisições por slot durante a janela
        'base_rate': 0,   # requisições por slot fora da janela
        'bit_options': [24, 32, 48, 64],
    },
}

print('Políticas:', policies)
print('Perfis de carga:', list(load_profiles.keys()))
print('Repetições por combinação:', trials_per_setting)

Políticas: ['threshold', 'on_demand', 'hybrid']
Perfis de carga: ['moderada', 'sobrecarga', 'burst']
Repetições por combinação: 5


## Execucao

Cada repeticao executa o fluxo de consumo via `controller.handle_key_request(...)`, que internamente usa `request_key_from_buffer()` e atualiza as metricas de pedido/consumo/falha do enlace.

In [ ]:
def build_request_schedule(profile: dict, rng: random.Random) -> list[tuple[int, int, int]]:
    schedule: list[tuple[int, int, int]] = []
    if profile['mode'] == 'uniform':
        for alice_id, bob_id in link_pairs:
            for _ in range(int(profile['requests_per_link'])):
                schedule.append((alice_id, bob_id, int(rng.choice(profile['bit_options']))))
        return schedule

    if profile['mode'] == 'burst_windows':
        duration = int(profile['duration_slots'])
        windows = [(int(a), int(b)) for (a, b) in profile['burst_windows']]
        burst_rate = int(profile['burst_rate'])
        base_rate = int(profile['base_rate'])
        bit_options = profile['bit_options']

        def in_window(t: int) -> bool:
            return any(a <= t < b for (a, b) in windows)

        for alice_id, bob_id in link_pairs:
            for t in range(duration):
                rate = burst_rate if in_window(t) else base_rate
                for _ in range(rate):
                    schedule.append((alice_id, bob_id, int(rng.choice(bit_options))))
        return schedule

    raise ValueError(f"Modo de carga desconhecido: {profile['mode']}")


mp_start_method = 'spawn'  # mais estável em notebooks
n_jobs = max(1, min(8, (os.cpu_count() or 2) - 1))
print(f"Multiprocessing: n_jobs={n_jobs} start_method={mp_start_method}")

tasks = []
for policy in policies:
    for load_name, profile in load_profiles.items():
        for trial in range(1, trials_per_setting + 1):
            schedule_seed = stable_seed(base_seed, 'exp2_schedule', policy, load_name, trial)
            rng = random.Random(schedule_seed)
            request_schedule = build_request_schedule(profile, rng)
            tasks.append({
                'policy': policy,
                'trial_id': trial,
                'base_seed': base_seed,
                'topology_name': 'Linha',
                'topology_nodes': 4,
                'link_pairs': link_pairs,
                'request_schedule': request_schedule,
                'min_threshold_bits': minimum_stock_bits,
                'buffer_capacity_bits': buffer_capacity_bits,
                'extra': {
                    'experiment': 'exp2',
                    'load': load_name,
                    'min_threshold_bits': minimum_stock_bits,
                    'buffer_capacity_bits': buffer_capacity_bits,
                    'schedule_seed': int(schedule_seed),
                    'schedule_requests_per_link_equiv': len(request_schedule) // max(1, len(link_pairs)),
                },
            })

rows = run_trials(tasks, n_jobs=n_jobs, mp_start_method=mp_start_method)

results_df = pd.DataFrame(rows)
display(results_df.sort_values(['policy', 'load', 'trial']).reset_index(drop=True))

summary_rows = aggregate_trials(
    rows,
    group_keys=['policy', 'load'],
    metric_keys=['service_rate', 'denial_rate', 'buffer_util_mean', 'replenishment_events', 'efficiency'],
 )
summary_df = pd.DataFrame(summary_rows)

load_order = {'moderada': 1, 'sobrecarga': 2, 'burst': 3}
summary_df['load_rank'] = summary_df['load'].map(load_order)
summary_df = summary_df.sort_values(['policy', 'load_rank']).reset_index(drop=True)

print('\nResumo (média ± desvio padrão)')
display(summary_df)

# Exporta CSVs
out_dir = Path('results') / f'validate_{trials_per_setting}'
out_dir.mkdir(parents=True, exist_ok=True)
write_csv(str(out_dir / 'exp2_trials.csv'), rows)
write_csv(str(out_dir / 'exp2_summary.csv'), summary_rows)
print(f"CSVs salvos em: {out_dir.resolve()}")

# Gráficos comparativos por política (linhas por perfil de carga)
metric_labels = {
    'service_rate': 'Taxa de atendimento',
    'denial_rate': 'Taxa de falha/negação',
    'buffer_util_mean': 'Utilização média do buffer',
    'replenishment_events': 'Eventos de reposição',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (metric, label) in zip(axes, metric_labels.items()):
    for policy in policies:
        subset = summary_df[summary_df['policy'] == policy].sort_values('load_rank')
        x = subset['load']
        y = subset[f'{metric}_mean']
        yerr = subset[f'{metric}_std']
        ax.errorbar(x, y, yerr=yerr, marker='o', capsize=4, label=policy)
    ax.set_title(label)
    ax.set_xlabel('Perfil de carga')
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)
    ax.legend(title='Política')

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

## Resultado detalhado por repeticao

In [ ]:
display(results_df.sort_values(['policy', 'load', 'trial']).reset_index(drop=True))

,policy,load,trial,total_generated_bits,total_consumed_bits,served_requests,denied_requests,replenishment_events,bits_available,service_rate,efficiency
0,hybrid,alta,1,656,656,8,22,10,0,0.266667,1.000000
1,hybrid,alta,2,608,608,10,20,10,0,0.333333,1.000000
2,hybrid,alta,3,640,640,9,21,10,0,0.300000,1.000000
3,hybrid,alta,4,672,624,12,18,12,48,0.400000,0.928571
4,hybrid,alta,5,608,608,9,21,8,0,0.300000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
67,threshold,media,4,192,192,4,14,2,0,0.222222,1.000000
68,threshold,media,5,288,256,6,12,3,32,0.333333,0.888889
69,threshold,media,6,288,208,5,13,3,80,0.277778,0.722222
70,threshold,media,7,192,160,4,14,2,32,0.222222,0.833333


## Resumo agregado por politica e carga

Media e desvio padrao das metricas principais para enxergar tendencia de saturacao.

In [ ]:
# (Opcional) Reexibir o mesmo resumo calculado acima
display(summary_df)

,policy,load,service_rate_mean,service_rate_std,efficiency_mean,efficiency_std,replenishment_events_mean,bits_available_mean,denied_requests_mean,served_requests_mean
0,hybrid,alta,0.325000,0.049602,0.985119,0.028279,10.125,12.0,20.250,9.750
1,hybrid,baixa,0.861111,0.115011,0.830640,0.152901,6.375,46.0,1.250,7.750
2,hybrid,media,0.638889,0.122438,0.929433,0.084766,10.250,34.0,6.500,11.500
3,on_demand,alta,0.283333,0.102353,1.000000,0.000000,8.500,0.0,21.500,8.500
4,on_demand,baixa,0.847222,0.144719,1.000000,0.000000,7.625,0.0,1.375,7.625
5,on_demand,media,0.631944,0.078216,1.000000,0.000000,11.375,0.0,6.625,11.375
6,threshold,alta,0.058333,0.061075,0.645833,0.421990,1.500,16.0,28.250,1.750
7,threshold,baixa,0.361111,0.129441,0.442708,0.140945,1.875,107.0,5.750,3.250
8,threshold,media,0.187500,0.106812,0.706597,0.344988,1.875,36.0,14.625,3.375


## Leitura rapida: ponto de saturacao

Critrio simples usado aqui: considerar saturacao quando `service_rate_mean < 0.90`.

Se nenhuma carga cair abaixo disso, a politica sustentou o intervalo testado.

In [ ]:
load_order = {'moderada': 1, 'sobrecarga': 2, 'burst': 3}
summary_ordered = summary_df.assign(load_rank=summary_df['load'].map(load_order)).sort_values(['policy', 'load_rank'])

print('Resumo por politica:')
for policy in policies:
    subset = summary_ordered[summary_ordered['policy'] == policy]

    saturation_rows = subset[subset['service_rate_mean'] < 0.90]
    if len(saturation_rows) > 0:
        sat_load = saturation_rows.iloc[0]['load']
    else:
        sat_load = 'nao saturou no intervalo testado'

    print(f'\n- Politica: {policy}')
    print(f'  Primeiro sinal de saturacao: {sat_load}')

    for _, row in subset.iterrows():
        util = row.get('buffer_util_mean_mean', None)
        util_str = 'n/a' if util is None or pd.isna(util) else f"{float(util):.3f}"
        print(
            f"  Carga {row['load']}: "
            f"atendimento={row['service_rate_mean']:.3f}, "
            f"eficiencia={row['efficiency_mean']:.3f}, "
            f"reposicoes={row['replenishment_events_mean']:.1f}, "
            f"util_buffer={util_str}"
        )

Resumo por politica:

- Politica: threshold
  Primeiro sinal de saturacao: baixa
  Carga baixa: atendimento=0.361, eficiencia=0.443, reposicoes=1.9, buffer_final=107.0
  Carga media: atendimento=0.188, eficiencia=0.707, reposicoes=1.9, buffer_final=36.0
  Carga alta: atendimento=0.058, eficiencia=0.646, reposicoes=1.5, buffer_final=16.0

- Politica: on_demand
  Primeiro sinal de saturacao: baixa
  Carga baixa: atendimento=0.847, eficiencia=1.000, reposicoes=7.6, buffer_final=0.0
  Carga media: atendimento=0.632, eficiencia=1.000, reposicoes=11.4, buffer_final=0.0
  Carga alta: atendimento=0.283, eficiencia=1.000, reposicoes=8.5, buffer_final=0.0

- Politica: hybrid
  Primeiro sinal de saturacao: baixa
  Carga baixa: atendimento=0.861, eficiencia=0.831, reposicoes=6.4, buffer_final=46.0
  Carga media: atendimento=0.639, eficiencia=0.929, reposicoes=10.2, buffer_final=34.0
  Carga alta: atendimento=0.325, eficiencia=0.985, reposicoes=10.1, buffer_final=12.0


## O que este experimento responde

Este cenario mostra em que ponto cada politica deixa de sustentar a rede QKD sob aumento de demanda, observando simultaneamente:
- queda de taxa de atendimento,
- mudanca na eficiencia de uso dos bits gerados,
- crescimento de eventos de reposicao,
- e nivel final de escassez residual no buffer.